In [1]:
import numpy as np
import pandas as pd

from popt.config import sect, intn, bond, metl, comd, universe
from popt.config import _2W, _1M, _3M, _6M, _1Y, _2Y, _3Y
from popt.alpha.modules.features import (
    momentum,
    momentum_vs,
    drawdown, 
    volatility, 
    volatility_downside, 
    sharpe_like, 
    skewness,
    kurtosis,
)

from popt.alpha.modules.features import FeatureBuilder, FeatureView
from popt.alpha.modules.predictor import RidgeRanker
from popt.alpha.modules.simulator import AlphaSimulator

from popt.backtest.modules.backtestdata import DataBuilder, DataLoader
from popt.backtest.modules.strategies import MetaStrategy, FixedWeights, Markowitz, asset_plot
from popt.backtest.modules.simulator import BacktestSimulator, wealth_plot
from popt.backtest.modules.riskmodel import RiskModel

In [2]:
fctr = ["SPY"]
ETF_CATEGORY = sect

TARGET = "mom_1m"
REGRESS = True   # NOTE NOTE NOTE NOTE NOTE NOTE NOTE NOTE NOTE NOTE NOTE NOTE

LOOKBACK = _1Y
HALFLIFE = _1Y
GAMMA = 100

FIRST_DATE = "2005-01-03"
FINAL_DATE = "2024-12-31"
SPREAD = 5e-4
LEVERAGE = 0.0
VC_LIMIT = 0.08
REBAL_FREQ = "M"

D0 = "1993-01-29"
D1 = "2026-03-24"
riskmodel = RiskModel.load_from_npz("../../data/riskmodel/k10_lb6m_hc6m_hv2m.npz")
rd = pd.read_parquet("../../data/return/return_d.parquet")#.loc[D0:D1]
rf = pd.read_parquet("../../data/return/ffr_d.parquet").reindex(rd.index)
rx = rd - rf.values

bsim = BacktestSimulator(SPREAD)

In [3]:
fb = FeatureBuilder(ret_d=rx, tickers=ETF_CATEGORY, factors=fctr, lookback=_1Y, first_date=D0, final_date=D1)
# add separate methods for adding constant and non-return features (macro and similar?)
# add feature of last days return
# add feature of last weeks returns

fb.add_feature(name="mom_2w", regress=REGRESS, z_scale=True, lookback=_2W, callback=momentum)
fb.add_feature(name="mom_1m", regress=REGRESS, z_scale=True, lookback=_1M, callback=momentum)
fb.add_feature(name="mom_3m", regress=REGRESS, z_scale=True, lookback=_3M, callback=momentum)
fb.add_feature(name="mom_6m", regress=REGRESS, z_scale=True, lookback=_6M, callback=momentum)
fb.add_feature(name="mom_1y", regress=REGRESS, z_scale=True, lookback=_1Y, callback=momentum)
fb.add_feature(name="mom_2y", regress=REGRESS, z_scale=True, lookback=_2Y, callback=momentum)
fb.add_feature(name="mom_3y", regress=REGRESS, z_scale=True, lookback=_3Y, callback=momentum)

fb.add_feature(name="mom_vs_2w", regress=REGRESS, z_scale=True, lookback=_2W, callback=momentum_vs)
fb.add_feature(name="mom_vs_1m", regress=REGRESS, z_scale=True, lookback=_1M, callback=momentum_vs)
fb.add_feature(name="mom_vs_3m", regress=REGRESS, z_scale=True, lookback=_3M, callback=momentum_vs)
fb.add_feature(name="mom_vs_6m", regress=REGRESS, z_scale=True, lookback=_6M, callback=momentum_vs)
fb.add_feature(name="mom_vs_1y", regress=REGRESS, z_scale=True, lookback=_1Y, callback=momentum_vs)
fb.add_feature(name="mom_vs_2y", regress=REGRESS, z_scale=True, lookback=_2Y, callback=momentum_vs)
fb.add_feature(name="mom_vs_3y", regress=REGRESS, z_scale=True, lookback=_3Y, callback=momentum_vs)

fb.add_feature(name="mdd_2w", regress=REGRESS, z_scale=True, lookback=_2W, callback=drawdown)
fb.add_feature(name="mdd_1m", regress=REGRESS, z_scale=True, lookback=_1M, callback=drawdown)
fb.add_feature(name="mdd_3m", regress=REGRESS, z_scale=True, lookback=_3M, callback=drawdown)
fb.add_feature(name="mdd_6m", regress=REGRESS, z_scale=True, lookback=_6M, callback=drawdown)
fb.add_feature(name="mdd_1y", regress=REGRESS, z_scale=True, lookback=_1Y, callback=drawdown)
fb.add_feature(name="mdd_2y", regress=REGRESS, z_scale=True, lookback=_2Y, callback=drawdown)
fb.add_feature(name="mdd_3y", regress=REGRESS, z_scale=True, lookback=_3Y, callback=drawdown)

fb.add_feature(name="vol_2w", regress=REGRESS, z_scale=True, lookback=_2W, callback=volatility)
fb.add_feature(name="vol_1m", regress=REGRESS, z_scale=True, lookback=_1M, callback=volatility)
fb.add_feature(name="vol_3m", regress=REGRESS, z_scale=True, lookback=_3M, callback=volatility)
fb.add_feature(name="vol_6m", regress=REGRESS, z_scale=True, lookback=_6M, callback=volatility)
fb.add_feature(name="vol_1y", regress=REGRESS, z_scale=True, lookback=_1Y, callback=volatility)
fb.add_feature(name="vol_2y", regress=REGRESS, z_scale=True, lookback=_2Y, callback=volatility)
fb.add_feature(name="vol_3y", regress=REGRESS, z_scale=True, lookback=_3Y, callback=volatility)

fb.add_feature(name="skw_2w", regress=REGRESS, z_scale=True, lookback=_2W, callback=skewness)
fb.add_feature(name="skw_1m", regress=REGRESS, z_scale=True, lookback=_1M, callback=skewness)
fb.add_feature(name="skw_3m", regress=REGRESS, z_scale=True, lookback=_3M, callback=skewness)
fb.add_feature(name="skw_6m", regress=REGRESS, z_scale=True, lookback=_6M, callback=skewness)
fb.add_feature(name="skw_1y", regress=REGRESS, z_scale=True, lookback=_1Y, callback=skewness)
fb.add_feature(name="skw_2y", regress=REGRESS, z_scale=True, lookback=_2Y, callback=skewness)
fb.add_feature(name="skw_3y", regress=REGRESS, z_scale=True, lookback=_3Y, callback=skewness)

fb.add_feature(name="krt_2w", regress=True, z_scale=True, lookback=_2W, callback=kurtosis)
fb.add_feature(name="krt_1m", regress=True, z_scale=True, lookback=_1M, callback=kurtosis)
fb.add_feature(name="krt_3m", regress=True, z_scale=True, lookback=_3M, callback=kurtosis)
fb.add_feature(name="krt_6m", regress=True, z_scale=True, lookback=_6M, callback=kurtosis)
fb.add_feature(name="krt_1y", regress=True, z_scale=True, lookback=_1Y, callback=kurtosis)
fb.add_feature(name="krt_2y", regress=True, z_scale=True, lookback=_2Y, callback=kurtosis)
fb.add_feature(name="krt_3y", regress=True, z_scale=True, lookback=_3Y, callback=kurtosis)

fb.consolidate()

In [4]:
fv = FeatureView(fb, target="mom_1m", subset=["mom_3m"])
fv.x.shape

(8344, 9, 1)

In [6]:
from itertools import combinations
fv = FeatureView(fb, target="mom_1m", subset=None)


for i, (f1, f2) in enumerate(combinations(fv.features, r=2)):
    fv = FeatureView(fb, target="mom_1m", subset=[f1, f2])

    rr = RidgeRanker(lookback=LOOKBACK, halflife=HALFLIFE, gamma=GAMMA)
    asim = AlphaSimulator(fv)
    asim.run(predictor=rr, verbose=False, permute=False)
    alpha = asim.get_alpha(universe)

    # alpha[:] = 1.0
    # alpha[:] = np.random.random(alpha.shape)

    db = DataBuilder(
        universe=rd.columns,
        first_date=FIRST_DATE,
        final_date=FINAL_DATE,
        alpha=alpha,
        rd=rd,
        rf=rf,
        riskmodel=riskmodel,
        rebal_freq=REBAL_FREQ,
    )
    dl = DataLoader(db=db, tickers=ETF_CATEGORY)
    m = Markowitz(
        dl=dl,
        lookahead=0,
        gamma=SPREAD,
        lev=LEVERAGE,
        w_max=0.7*np.ones(dl.N),
        vc_lim=VC_LIMIT,
    )
    bsim.run(strategy=m, verbose=False)

    sharpe = np.round(bsim.ann_sharpe, decimals=4)
    spearm = np.round(np.nanmean(asim.ic_spearman), decimals=4)
    pearsn = np.round(np.nanmean(asim.ic_pearson), decimals=4)
    # thetas = asim.thetas[:,i]
    # u_theta = np.nanmean(thetas).round(4)
    # s_theta = np.nanstd(thetas).round(4)

    print(f1, f2, sharpe, spearm, pearsn)
    # wealth_plot(bsim)

# NOTE without rank post processing
# sect -> 0.5824 0.0168 krt_3y (0.5012) raw
# bond -> 0.3687 -0.0375 skw_3y (0.2774) raw
# metl -> 0.4892 0.0792 mom_vs_3m (0.3951) raw
# intn -> 0.4095 0.0297 mom_2w (0.0854) raw
# comd -> -0.0155 -0.0043 mdd_2w (-0.2014) raw

# mom_3m skw_1m 0.6662 -0.0034 0.001
# mom_3m skw_3m 0.5195 -0.0159 -0.0183
# mom_3m skw_6m 0.6787 0.0107 0.0198
# mom_3m skw_1y 0.6622 0.0132 0.0217

# mdd_1y skw_2w 0.6701 0.027 0.0229
# mdd_1y skw_1m 0.5326 0.0241 0.0204
# mdd_1y skw_3m 0.7536 0.0188 0.0156  # NOTE
# mdd_1y skw_6m 0.6203 0.016 0.0077
# mdd_1y skw_1y 0.5588 0.03 0.0165
# mdd_1y skw_2y 0.5925 0.0125 0.011
# mdd_1y skw_3y 0.3975 0.0181 0.017
# mdd_1y krt_2w 0.7234 0.0349 0.0359
# mdd_1y krt_1m 0.7016 0.0385 0.0363
# mdd_1y krt_3m 0.5929 0.0431 0.0329
# mdd_1y krt_6m 0.5811 0.0308 0.0177
# mdd_1y krt_1y 0.6697 0.0252 0.032

mom_2w mom_1m 0.3731 -0.0203 -0.015
mom_2w mom_3m 0.4278 -0.0023 0.0047
mom_2w mom_6m 0.3468 0.0111 0.0107
mom_2w mom_1y 0.3454 0.1037 0.1028
mom_2w mom_2y 0.3448 0.1016 0.1126
mom_2w mom_3y 0.2806 0.0869 0.0955
mom_2w mom_vs_2w 0.2477 -0.0089 -0.0071
mom_2w mom_vs_1m 0.3978 -0.0205 -0.0147
mom_2w mom_vs_3m 0.4248 -0.003 0.0024
mom_2w mom_vs_6m 0.3596 0.0165 0.0173
mom_2w mom_vs_1y 0.388 0.1167 0.1212
mom_2w mom_vs_2y 0.3877 0.1163 0.1312
mom_2w mom_vs_3y 0.3812 0.0911 0.1034
mom_2w mdd_2w 0.4158 -0.0363 -0.0421
mom_2w mdd_1m 0.4059 -0.0241 -0.0226
mom_2w mdd_3m 0.4814 -0.009 -0.0086
mom_2w mdd_6m 0.3279 -0.0207 -0.032
mom_2w mdd_1y 0.4745 0.0205 0.0166
mom_2w mdd_2y 0.3888 -0.0042 -0.0099
mom_2w mdd_3y 0.3335 -0.0252 -0.036
mom_2w vol_2w 0.346 -0.024 -0.0318
mom_2w vol_1m 0.3295 -0.0304 -0.0333
mom_2w vol_3m 0.3404 -0.0303 -0.0383
mom_2w vol_6m 0.4168 -0.0299 -0.0385
mom_2w vol_1y 0.4362 -0.035 -0.0452
mom_2w vol_2y 0.3929 -0.024 -0.0299
mom_2w vol_3y 0.4121 -0.026 -0.0334
mom_2w skw_

In [9]:
sharpes = pd.read_csv("../../.deprecated/sect_2f.csv")
sharpes.sort_values("sharpe", ascending=False).head(25)

,f1,f2,sharpe,spearm,pearsn
596,mdd_1y,skw_3m,0.7536,0.0188,0.0156
601,mdd_1y,krt_2w,0.7234,0.0349,0.0359
602,mdd_1y,krt_1m,0.7016,0.0385,0.0363
109,mom_3m,skw_6m,0.6787,0.0107,0.0198
592,mdd_1y,vol_2y,0.6731,0.0126,0.0018
594,mdd_1y,skw_2w,0.6701,0.0270,0.0229
605,mdd_1y,krt_1y,0.6697,0.0252,0.0320
107,mom_3m,skw_1m,0.6662,-0.0034,0.0010
525,mdd_1m,skw_6m,0.6635,-0.0057,-0.0033
110,mom_3m,skw_1y,0.6622,0.0132,0.0217


In [ ]:
# N = 500
# sharpes_rand_perm = []
# for i in range(N):
#     fv = FeatureView(fb, target="mom_1m", subset=None)
#     fv.add_mask(fv.tickers, ["krt_2w"], exclude=False)
#     fv.apply_masking()

#     rr = RidgeRanker(lookback=LOOKBACK, halflife=HALFLIFE, gamma=GAMMA)
#     asim = AlphaSimulator(fv)
#     asim.run(predictor=rr, verbose=False, permute=True)

#     tickers = asim.fv.tickers
#     timeline = asim.fv.timeline
#     prd = asim.prd.copy()
#     T = prd.shape[0]
#     U = len(universe)
#     alpha = np.full((T,U), fill_value=np.nan, dtype=float)
#     i_N = np.array([universe.index(t) for t in tickers], dtype=int)
#     alpha[:,i_N] = prd
#     alpha = pd.DataFrame(data=alpha, columns=universe, index=timeline)
#     # alpha = alpha.rank(axis=1)
#     alpha = alpha.fillna(1.0)

#     # alpha[:] = 1.0
#     # alpha[:] = np.random.random(alpha.shape)

#     db = DataBuilder(
#         universe=rd.columns,
#         first_date=FIRST_DATE,
#         final_date=FINAL_DATE,
#         alpha_d=alpha,
#         return_d=rd,
#         rf_d=rf,
#         riskmodel=riskmodel,
#         rebal_freq=REBAL_FREQ,
#     )
#     dl = DataLoader(db=db, tickers=ETF_CATEGORY)
#     m = Markowitz(
#         dl=dl,
#         lookahead=0,
#         gamma=SPREAD,
#         lev=LEVERAGE,
#         w_max=0.7*np.ones(dl.N),
#         vc_lim=VC_LIMIT,
#     )
#     bsim.run(strategy=m, verbose=False)

#     sharpe = np.round(bsim.ann_sharpe, decimals=4)
#     spearm = np.round(np.nanmean(asim.ic_spearman), decimals=4)
#     pearsn = np.round(np.nanmean(asim.ic_pearson), decimals=4)

#     # print("mean theta", np.nanmean(asim.thetas, axis=0)[i])
#     print(i, sharpe)
#     sharpes_rand_perm.append(sharpe)
#     # wealth_plot(bsim)

# # NOTE without rank post processing
# # sect -> 0.5824 0.0168 krt_3y (0.5012) raw
# # bond -> 0.3687 -0.0375 skw_3y (0.2774) raw
# # metl -> 0.4892 0.0792 mom_vs_3m (0.3951) raw
# # intn -> 0.4095 0.0297 mom_2w (0.0854) raw
# # comd -> -0.0155 -0.0043 mdd_2w (-0.2014) raw

# # 0.6550
# # p_val = (1 + np.sum(np.array(sharpes_rand_perm) >= 0.6550)) / (N+1)
# # p_val
# p val : 0.031936127744510975